# Tugas 1 — Milestone 3
## Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Nama Mahasiswa:** Rizqullah Zuhayr Alfito Khan  
**NIM:** 202310370311124  
**Username GitHub:** https://github.com/alfitXX

### Tujuan

Milestone 3 bertujuan untuk melakukan analisis lanjutan terhadap dataset
yang telah melalui proses data cleaning pada Milestone 2.

Analisis yang dilakukan meliputi:
1. Analisis temporal dan windowing.
2. Analisis distribusi data.
3. Analisis korelasi antar variabel.
4. Analisis kategori.
5. Analisis pola temporal.
6. Analisis multi-dimensi menggunakan visualisasi interaktif.
7. Perumusan insight berdasarkan hasil analisis.

## Bagian 1: Inisialisasi dan Memuat Data Bersih

Muat dataset hasil pembersihan dari folder `data/` menggunakan Polars.

In [1]:
import duckdb
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

In [ ]:
# Path dataset hasil cleaning dari Milestone 2
path_data = Path("../data/processed/dataset_bersih.parquet")

# Membaca dataset menggunakan Polars
df = pl.read_parquet(path_data)

print(f"Data loaded: {df.height:,} baris x {df.width} kolom")
print("\nKolom dataset:")
print(df.columns)

In [ ]:
print("Schema dataset:")
print(df.schema)

print("\n5 data pertama:")
display(df.head())

In [ ]:
print("Statistik deskriptif dataset:")
display(df.describe())

## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)

Terapkan teknik windowing Polars (`group_by_dynamic`, `rolling_mean`, `shift`, atau DuckDB window functions)
untuk menangkap tren waktu, pola musiman, atau pergerakan akumulatif.

In [ ]:
# Mencari kandidat kolom tanggal/waktu
time_candidates = [
    "date",
    "tanggal",
    "datetime",
    "timestamp",
    "published_at",
    "publish_date",
    "created_at",
    "time",
    "waktu"
]

time_col = None

# Pencarian berdasarkan nama kolom
for col in df.columns:
    if col.lower() in time_candidates:
        time_col = col
        break

# Jika belum ditemukan, cari berdasarkan kata dalam nama kolom
if time_col is None:
    for col in df.columns:
        nama = col.lower()
        if any(kata in nama for kata in [
            "date", "tanggal", "time", "timestamp",
            "publish", "created"
        ]):
            time_col = col
            break

print("Kolom waktu yang digunakan:", time_col)

In [ ]:
if time_col is not None:
    df = df.with_columns(
        pl.col(time_col)
        .cast(pl.String)
        .str.to_datetime(strict=False)
        .alias("timestamp")
    )

    print("Kolom timestamp berhasil dibuat.")

    print(
        "Jumlah data dengan timestamp valid:",
        df.select(
            pl.col("timestamp").is_not_null().sum()
        ).item()
    )
else:
    print("Kolom tanggal/waktu tidak ditemukan.")

In [ ]:
if "timestamp" in df.columns:

    df_temporal = (
        df
        .filter(pl.col("timestamp").is_not_null())
        .group_by_dynamic(
            "timestamp",
            every="1d"
        )
        .agg(
            pl.len().alias("jumlah_berita")
        )
        .sort("timestamp")
    )

    print("Data temporal per hari:")
    display(df_temporal.head(10))

else:
    print("Analisis temporal tidak dapat dilakukan karena kolom timestamp tidak tersedia.")

In [ ]:
group_by_dynamic()

In [ ]:
if df_temporal.height > 0:

    df_temporal = df_temporal.with_columns(
        pl.col("jumlah_berita")
        .rolling_mean(
            window_size=7,
            min_samples=1
        )
        .alias("moving_average_7_hari")
    )

    print("Moving average 7 hari berhasil dibuat.")
    display(df_temporal.head(10))

In [ ]:
if df_temporal.height > 0:

    df_temporal = df_temporal.with_columns(
        pl.col("jumlah_berita")
        .shift(1)
        .alias("jumlah_hari_sebelumnya")
    )

    df_temporal = df_temporal.with_columns(
        (
            pl.col("jumlah_berita")
            - pl.col("jumlah_hari_sebelumnya")
        ).alias("perubahan_harian")
    )

    display(df_temporal.head(10))

In [ ]:
shift()

## Bagian 3: Minimal 6 Visualisasi Interaktif (Plotly / Altair)

Bangun minimal 6 visualisasi interaktif yang mencakup:
- Visualisasi 1: Tren deret waktu (Line chart dengan range slider)
- Visualisasi 2: Distribusi variabel utama (Box plot / Histogram / Violin plot)
- Visualisasi 3: Korelasi antar-variabel numerik (Heatmap / Scatter plot sampling)
- Visualisasi 4: Komposisi kategori utama (Bar chart / Treemap)
- Visualisasi 5: Analisis siklus musiman / perbandingan kelompok
- Visualisasi 6: Pola spasial atau korelasi multi-dimensi

In [ ]:
# Visualisasi 1: Tren Waktu Interaktif

if df_temporal.height > 0:

    temporal_pd = df_temporal.to_pandas()

    fig1 = px.line(
        temporal_pd,
        x="timestamp",
        y=[
            "jumlah_berita",
            "moving_average_7_hari"
        ],
        title="Tren Jumlah Berita dan Moving Average 7 Hari",
        labels={
            "timestamp": "Tanggal",
            "value": "Jumlah Berita",
            "variable": "Indikator"
        }
    )

    fig1.update_layout(
        xaxis=dict(
            rangeslider=dict(visible=True),
            rangeselector=dict(
                buttons=[
                    dict(
                        count=7,
                        label="7 Hari",
                        step="day",
                        stepmode="backward"
                    ),
                    dict(
                        count=1,
                        label="1 Bulan",
                        step="month",
                        stepmode="backward"
                    ),
                    dict(
                        step="all",
                        label="Semua"
                    )
                ]
            )
        ),
        hovermode="x unified"
    )

    fig1.show()

In [ ]:
# Visualisasi 2: Distribusi dan Outlier

if df_temporal.height > 0:

    fig2 = px.box(
        temporal_pd,
        y="jumlah_berita",
        points="outliers",
        title="Distribusi dan Outlier Jumlah Berita per Hari",
        labels={
            "jumlah_berita": "Jumlah Berita per Hari"
        }
    )

    fig2.show()

In [ ]:
# Visualisasi 3: Korelasi Antar-Variabel Numerik

numeric_cols = [
    col for col, dtype in df.schema.items()
    if dtype.is_numeric()
]

print("Kolom numerik:")
print(numeric_cols)

if len(numeric_cols) >= 2:

    corr_matrix = df.select(numeric_cols).corr()

    fig3 = px.imshow(
        corr_matrix,
        text_auto=True,
        aspect="auto",
        title="Heatmap Korelasi Antar Variabel Numerik"
    )

    fig3.show()

else:
    print(
        "Jumlah kolom numerik kurang dari 2, "
        "sehingga heatmap korelasi tidak dapat dibuat."
    )

In [ ]:
# Mencari kandidat kolom kategori

category_candidates = [
    "category",
    "kategori",
    "label",
    "topic",
    "jenis",
    "section",
    "rubrik"
]

category_col = None

for col in df.columns:
    if col.lower() in category_candidates:
        category_col = col
        break

if category_col is None:
    for col in df.columns:
        nama = col.lower()

        if any(kata in nama for kata in [
            "category",
            "kategori",
            "topic",
            "label",
            "rubrik"
        ]):
            category_col = col
            break

print("Kolom kategori:", category_col)

In [ ]:
# Visualisasi 4: Komposisi Kategori Utama

if category_col is not None:

    category_df = (
        df
        .filter(pl.col(category_col).is_not_null())
        .group_by(category_col)
        .agg(
            pl.len().alias("jumlah_berita")
        )
        .sort(
            "jumlah_berita",
            descending=True
        )
        .head(15)
    )

    category_pd = category_df.to_pandas()

    fig4 = px.bar(
        category_pd,
        x=category_col,
        y="jumlah_berita",
        title="15 Kategori Berita dengan Jumlah Terbanyak",
        labels={
            category_col: "Kategori",
            "jumlah_berita": "Jumlah Berita"
        }
    )

    fig4.update_layout(
        xaxis_tickangle=-45
    )

    fig4.show()

else:
    print("Kolom kategori tidak ditemukan.")

In [ ]:
# Membuat variabel hari dan jam

if "timestamp" in df.columns:

    df_seasonal = (
        df
        .filter(
            pl.col("timestamp").is_not_null()
        )
        .with_columns(
            [
                pl.col("timestamp")
                .dt.weekday()
                .alias("hari_minggu"),

                pl.col("timestamp")
                .dt.hour()
                .alias("jam")
            ]
        )
    )

    print("Data untuk analisis pola temporal berhasil dibuat.")
    display(
        df_seasonal
        .select([
            "timestamp",
            "hari_minggu",
            "jam"
        ])
        .head()
    )

In [ ]:
# Visualisasi 5: Pola Musiman

if "df_seasonal" in globals():

    seasonal_df = (
        df_seasonal
        .group_by([
            "hari_minggu",
            "jam"
        ])
        .agg(
            pl.len().alias("jumlah_berita")
        )
        .sort([
            "hari_minggu",
            "jam"
        ])
    )

    seasonal_pd = seasonal_df.to_pandas()

    fig5 = px.density_heatmap(
        seasonal_pd,
        x="jam",
        y="hari_minggu",
        z="jumlah_berita",
        histfunc="sum",
        title="Pola Publikasi Berita Berdasarkan Hari dan Jam",
        labels={
            "jam": "Jam",
            "hari_minggu": "Hari dalam Minggu",
            "jumlah_berita": "Jumlah Berita"
        }
    )

    fig5.show()

In [ ]:
# Visualisasi 6: Analisis Multi-Dimensi

if (
    category_col is not None
    and "timestamp" in df.columns
):

    multi_df = (
        df
        .filter(
            pl.col(category_col).is_not_null() &
            pl.col("timestamp").is_not_null()
        )
        .with_columns(
            pl.col("timestamp")
            .dt.date()
            .alias("tanggal")
        )
        .group_by([
            "tanggal",
            category_col
        ])
        .agg(
            pl.len().alias("jumlah_berita")
        )
        .sort(
            "jumlah_berita",
            descending=True
        )
        .head(1000)
    )

    multi_pd = multi_df.to_pandas()

    fig6 = px.scatter(
        multi_pd,
        x="tanggal",
        y="jumlah_berita",
        size="jumlah_berita",
        color=category_col,
        hover_data=[
            "tanggal",
            category_col,
            "jumlah_berita"
        ],
        title="Analisis Multi-Dimensi Jumlah Berita Berdasarkan Waktu dan Kategori",
        labels={
            "tanggal": "Tanggal",
            "jumlah_berita": "Jumlah Berita",
            category_col: "Kategori"
        }
    )

    fig6.show()

else:
    print(
        "Kolom timestamp atau kategori tidak tersedia."
    )

In [ ]:
print("===== HASIL ANALISIS TREN =====")

if df_temporal.height > 0:

    tertinggi = (
        df_temporal
        .sort(
            "jumlah_berita",
            descending=True
        )
        .head(1)
    )

    terendah = (
        df_temporal
        .sort(
            "jumlah_berita"
        )
        .head(1)
    )

    print("\nTanggal dengan jumlah berita tertinggi:")
    display(tertinggi)

    print("\nTanggal dengan jumlah berita terendah:")
    display(terendah)

    print("\nRata-rata jumlah berita per hari:")
    print(
        df_temporal
        .select(
            pl.col("jumlah_berita").mean()
        )
    )

In [ ]:
print("===== HASIL ANALISIS KATEGORI =====")

if category_col is not None:

    top_category = (
        df
        .filter(
            pl.col(category_col).is_not_null()
        )
        .group_by(category_col)
        .agg(
            pl.len().alias("jumlah_berita")
        )
        .sort(
            "jumlah_berita",
            descending=True
        )
        .head(10)
    )

    display(top_category)

In [ ]:
print("===== HASIL ANALISIS HARI =====")

if "timestamp" in df.columns:

    weekday_df = (
        df
        .filter(
            pl.col("timestamp").is_not_null()
        )
        .with_columns(
            pl.col("timestamp")
            .dt.strftime("%A")
            .alias("hari")
        )
        .group_by("hari")
        .agg(
            pl.len().alias("jumlah_berita")
        )
        .sort(
            "jumlah_berita",
            descending=True
        )
    )

    display(weekday_df)

## Bagian 4: Rumusan 5 Insight Analitik Utama

Tuliskan minimal 5 temuan atau wawasan penting yang Anda peroleh dari visualisasi di atas:

1. **Insight 1 (Tren Waktu & Musiman):**  
   [Jelaskan temuan terkait tren jangka panjang atau pola berulang yang teridentifikasi]

2. **Insight 2 (Distribusi & Anomali):**  
   [Jelaskan temuan terkait sebaran data, ketimpangan, atau keberadaan pencilan penting]

3. **Insight 3 (Korelasi Kunci):**  
   [Jelaskan hubungan signifikan antar-variabel yang dapat dijadikan dasar keputusan bisnis/kebijakan]

4. **Insight 4 (Perilaku Kategori / Segmen):**  
   [Jelaskan perbedaan karakteristik performa antar kelompok/wilayah/kategori produk]

5. **Insight 5 (Rekomendasi Praktis):**  
   [Jelaskan tindakan nyata atau strategi operasional yang direkomendasikan berdasarkan seluruh temuan di atas]
